In [1]:
import pandas as pd
import torch.nn as nn
import torch
import re
from sklearn.model_selection import train_test_split
from collections import Counter
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset

In [2]:
df = pd.read_csv("../data/raw/bbc-text.csv")

In [3]:
df.head()

,category,text
0,tech,tv future in the hands of viewers with home th...
1,business,worldcom boss left books alone former worldc...
2,sport,tigers wary of farrell gamble leicester say ...
3,sport,yeading face newcastle in fa cup premiership s...
4,entertainment,ocean s twelve raids box office ocean s twelve...


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   category  2225 non-null   str  
 1   text      2225 non-null   str  
dtypes: str(2)
memory usage: 34.9 KB


In [5]:
df.nunique()

category       5
text        2126
dtype: int64

In [6]:
df["category"].value_counts(normalize=True)*100

category
sport            22.966292
business         22.921348
politics         18.741573
tech             18.022472
entertainment    17.348315
Name: proportion, dtype: float64

In [7]:
for val in df["category"].unique().tolist():
    display(df[df["category"] == val][["category", "text"]].sample(4, random_state=42))

,category,text
1605,tech,sony psp console hits us in march us gamers wi...
1592,tech,iran jails blogger for 14 years an iranian web...
234,tech,itunes user sues apple over ipod a user of app...
1206,tech,speak easy plan for media players music and fi...


,category,text
2098,business,irish company hit by iraqi report shares in ir...
1972,business,bat spit drug firm goes to market a german fir...
2070,business,electronics firms eye plasma deal consumer ele...
1930,business,circuit city gets takeover offer circuit city ...


,category,text
517,sport,ireland 17-12 south africa ronan o gara scored...
360,sport,henman & murray claim lta awards tim henman wa...
1889,sport,moya suffers shock loss fifth seed carlos moya...
1099,sport,everton s weir cools euro hopes everton defend...


,category,text
1871,entertainment,singer ferguson facing eviction three degree...
1686,entertainment,police praise courageous ozzy rock star ozzy...
555,entertainment,the sound of music is coming home the original...
1490,entertainment,farrell due to make us tv debut actor colin fa...


,category,text
827,politics,brown outlines third term vision gordon brown ...
2101,politics,abbas will not tolerate attacks palestinian ...
2197,politics,campbell returns to election team ex-downing s...
486,politics,kennedy looks to election gains they may not k...


In [8]:
df["new_text"] = df["text"].str.lower().str.replace(r"\s+", " ", regex=True).str.replace(r"[^\w\s]", " ", regex=True).str.strip()
#df["new_text"] = df["text"].str.lower().str.replace(".", " ").str.replace(r"[^\w\s]", " ", regex=True).str.strip()
df = df.drop_duplicates(subset="new_text").copy()
df.info()

<class 'pandas.DataFrame'>
Index: 2123 entries, 0 to 2224
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   category  2123 non-null   str  
 1   text      2123 non-null   str  
 2   new_text  2123 non-null   str  
dtypes: str(3)
memory usage: 66.3 KB


In [9]:
len(df["text"] != df["new_text"])

2123

In [10]:
X = df["new_text"]
y = df["category"]

In [11]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [12]:
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val)

In [13]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

PAD_ID = 0
UNK_ID = 1

vocab = {
    PAD_TOKEN: PAD_ID,
    UNK_TOKEN: UNK_ID
}

label_to_id = {
    label: idx
    for idx, label in enumerate(sorted(y_train.unique()))
}

id_to_label = {
    idx: label
    for label, idx in label_to_id.items()
}

word_counts = Counter(
    word
    for text in X_train
    for word in text.split()
)

for word in word_counts:
    vocab[word] = len(vocab)

In [14]:
print("Classes: ", label_to_id)
print("\nNumber of classes: ", len(label_to_id))
print("\nVocabulary size: ", len(vocab))

Classes:  {'business': 0, 'entertainment': 1, 'politics': 2, 'sport': 3, 'tech': 4}

Number of classes:  5

Vocabulary size:  23786


In [15]:
PAD_ID = vocab[PAD_TOKEN]
UNK_ID = vocab[UNK_TOKEN]

def tokenize_encode(txt, vocab):
    tokens = txt.split()
    tokens_id = [vocab.get(token, UNK_ID) for token in tokens]
    return tokens, tokens_id
                 
# sample_text = X_train

# tokens, tokens_id = tokenize_encode(sample_text, vocab)

X_train_token_ids = [
    tokenize_encode(text, vocab)[1]
    for text in X_train
]

X_val_token_ids = [
    tokenize_encode(text, vocab)[1]
    for text in X_val
]

X_test_token_ids = [
    tokenize_encode(text, vocab)[1]
    for text in X_test
]

print((X_test_token_ids[:2]))

[[31, 385, 2174, 34, 67, 8608, 31, 385, 13430, 95, 4346, 37, 4330, 1514, 279, 1095, 5776, 34, 59, 1505, 7, 809, 221, 555, 118, 8608, 44, 15, 4628, 4404, 11, 4237, 73, 5439, 73, 37, 1462, 4330, 4346, 1641, 147, 252, 7, 13430, 1500, 367, 84, 44, 124, 7, 2012, 1404, 271, 2835, 1322, 61, 7, 2297, 8608, 5230, 185, 1665, 2764, 2001, 3399, 899, 7, 127, 2595, 73, 7, 4330, 3449, 91, 7239, 32, 84, 44, 20658, 98, 540, 14322, 1665, 7, 13430, 4330, 86, 4346, 1514, 4274, 6445, 1, 37, 518, 1, 11614, 32, 1104, 4012, 1, 11, 6990, 323, 41, 70, 34, 59, 7, 555, 4955, 73, 185, 1573, 1, 5176, 1, 5176, 32, 1, 939, 50, 41, 6931, 11, 5509, 144, 55, 4404, 11, 4117, 2001, 3399, 2835, 1497, 1363, 1905, 59, 4622, 84, 1735, 10854, 13430, 1500, 367, 14017, 7, 2012, 1404, 1363, 15, 175, 11, 7, 266, 8180, 32, 17470, 225, 4499, 3774, 783, 7, 1500, 367, 5793, 50, 2835, 1322, 61, 368, 1, 32, 368, 1, 221, 1573, 5230, 7, 1149, 6018, 7, 10690, 11, 94, 1, 10803, 32, 899, 2375, 57, 1363, 271, 622, 271, 7, 13430, 597, 3399, 32

In [16]:
print("Train articles:", len(X_train_token_ids), len(X_train))
print("Validation articles:", len(X_val_token_ids), len(X_val))
print("Test articles:", len(X_test_token_ids), len(X_test))

Train articles: 1273 1273
Validation articles: 425 425
Test articles: 425 425


In [17]:
y_train_ids = [label_to_id[label] for label in y_train]
y_val_ids = [label_to_id[label] for label in y_val]
y_test_ids = [label_to_id[label] for label in y_test]

print("Train classes:", len(y_train_ids), len(y_train))
print("Validation classes:", len(y_val_ids), len(y_val))
print("Test classes:", len(y_test_ids), len(y_test))

Train classes: 1273 1273
Validation classes: 425 425
Test classes: 425 425


In [18]:
y_train_tensor = torch.tensor(y_train_ids, dtype=torch.long)
y_val_tensor = torch.tensor(y_val_ids, dtype=torch.long)
y_test_tensor = torch.tensor(y_test_ids, dtype=torch.long)

print("Label mapping:", label_to_id)
print("Train labels shape:", y_train_tensor.shape)
print("Example labels:", y_train_tensor[:10].tolist())

Label mapping: {'business': 0, 'entertainment': 1, 'politics': 2, 'sport': 3, 'tech': 4}
Train labels shape: torch.Size([1273])
Example labels: [4, 0, 2, 2, 4, 4, 3, 4, 2, 4]


In [19]:
train_length = [len(ids) for ids in X_train_token_ids]

print("Largest article: ", max(train_length), "tokens")
print("Shortest article: ", min(train_length), "tokens")
print("Average average: ", sum(train_length)/ len(train_length), "tokens")

Largest article:  4491 tokens
Shortest article:  91 tokens
Average average:  398.1366849960723 tokens


In [20]:
MAX_LENGTH = 512

X_train_token_ids = [
    ids[:MAX_LENGTH]
    for ids in X_train_token_ids
]

X_test_token_ids = [
    ids[:MAX_LENGTH]
    for ids in X_test_token_ids
]

X_val_token_ids = [
    ids[:MAX_LENGTH]
    for ids in X_val_token_ids
]

In [21]:
train_length = [len(ids) for ids in X_train_token_ids]

print("Largest article: ", max(train_length), "tokens")
print("Shortest article: ", min(train_length), "tokens")
print("Average article: ", sum(train_length)/ len(train_length), "tokens")

Largest article:  512 tokens
Shortest article:  91 tokens
Average article:  354.97879025923015 tokens


In [22]:
print(len(ids) == MAX_LENGTH for ids in X_train_token_ids)

<generator object <genexpr> at 0x11d7c2dc0>


In [23]:
X_train_tensor = [
    torch.tensor(ids, dtype=torch.long)
    for ids in X_train_token_ids
]

X_test_tensor = [
    torch.tensor(ids, dtype=torch.long)
    for ids in X_test_token_ids
]

X_val_tensor = [
    torch.tensor(ids, dtype=torch.long)
    for ids in X_val_token_ids
]

X_train_padded = pad_sequence(
    X_train_tensor,
    batch_first=True,
    padding_value=PAD_ID
)

X_test_padded = pad_sequence(
    X_test_tensor,
    batch_first=True,
    padding_value=PAD_ID
)

X_val_padded = pad_sequence(
    X_val_tensor,
    batch_first=True,
    padding_value=PAD_ID
)

print("Padded after training shape: ", X_train_padded.shape)
print("Padded after testing shape: ", X_test_padded.shape)
print("Padded after val shape: ", X_val_padded.shape)

Padded after training shape:  torch.Size([1273, 512])
Padded after testing shape:  torch.Size([425, 512])
Padded after val shape:  torch.Size([425, 512])


In [24]:
print("Train: ", X_train_padded.shape, y_train_tensor.shape)
print("Validation: ", X_val_padded.shape, y_val_tensor.shape)
print("Testing: ", X_test_padded.shape, y_test_tensor.shape)

Train:  torch.Size([1273, 512]) torch.Size([1273])
Validation:  torch.Size([425, 512]) torch.Size([425])
Testing:  torch.Size([425, 512]) torch.Size([425])


In [25]:
assert X_train_padded.size(0) == y_train_tensor.size(0)
assert X_val_padded.size(0) == y_val_tensor.size(0)
assert X_test_padded.size(0) == y_test_tensor.size(0)

In [26]:
class BBCTextClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, pad_id, embedding_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings= vocab_size,
            embedding_dim=embedding_dim,
            padding_idx= pad_id
        )

        self.classifier = nn.Linear(embedding_dim, num_classes)
        self.pad_id = pad_id

    def forward(self, input_ids):

        embedded = self.embedding(input_ids)

        mask = input_ids.ne(self.pad_id).unsqueeze(-1)
        mask = mask.to(embedded.dtype)

        summed_embeddings = (embedded*mask).sum(dim=(1))
        token_counts = mask.sum(dim=(1)).clamp_min(1)

        document_vector = summed_embeddings/ token_counts

        logits = self.classifier(document_vector)

        return logits
        

In [27]:
vocab_size = len(vocab)
embedding_dim = 64
num_classes = len(label_to_id)

model = BBCTextClassifier(
    vocab_size= vocab_size,
    num_classes = num_classes,
    pad_id = PAD_ID,
    embedding_dim = embedding_dim
)

sample_logits = model(X_train_padded[:4])
sample_labels = y_train_tensor[:4]

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [28]:
loss = criterion(sample_logits, sample_labels).item()
loss

1.6351509094238281

In [30]:
train_dataset = TensorDataset(X_train_padded, y_train_tensor)

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size= BATCH_SIZE,
    shuffle=True
)

print("Training batches: ", len(train_loader))

Training batches:  40
